# Rating curves: extras

Contents:

1. [Putting stage on the right datum](#datum)
2. [The stage of zero flow, and Johnson's method](#johnson)
3. [Sampler diagnostics](#diagnostics): R-hat, ESS, Pareto-k, and ADVI against NUTS
4. [Cross-validation, in depth](#cv-depth), including high-flow extrapolation
5. [The same sensor, from the MAGL files](#magl)
6. [The map](#map)
7. [Fitting many sites in parallel](#many)
8. [The command line](#cli)

Requires `PAGAIA_AUTH_TOKEN` environment variable: the setup cell reads
the SBR-09 station from pagaia, exactly as section 2 of `getting_started.ipynb` does.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import limnotech_rating_curves as lrc
from limnotech_rating_curves.data import magl
from fb_pagaia import core, utils

# Everything section 2 of getting_started.ipynb builds, rebuilt here so the sections
# below stand alone. Needs PAGAIA_AUTH_TOKEN and the VPN.
api = utils.start_session()
sbr09_station = core.Station.from_name(name="SBR-09_WL", api=api)
field_discharge = magl.discharge_measurements("SBR-09")[["time", "discharge_cfs"]]

sbr09_sample = lrc.station_sample(sbr09_station, discharge=field_discharge,
                                  stage_datum="lowest")                  # gage height
raw_axis = lrc.station_sample(sbr09_station, discharge=field_discharge)  # datum ignored

comparison = lrc.compare(sbr09_sample)
sbr09 = comparison["power_law"]      # the baseline NUTS fit, as in getting_started
best = comparison.best
cv = lrc.cross_validate(sbr09_sample, models=["power_law", "spline", "bdrc_gplm0"])

print(sbr09_sample)
print("best predictive score:", best.name)
print(cv)

<a id="datum"></a>
## 1. Putting stage on the right datum

Section 2 of `getting_started.ipynb` passed `stage_datum="lowest"` without
explaining it. This section is what that argument does, and why SBR-09 needs it at all.

A rating is fitted to gage height: the water surface above a fixed local reference that
sits low enough to keep the number positive. Sensors usually report something else.
SBR-09 reports a depth above the sensor's own zero, which is neither gage height nor an
elevation on any survey datum — it runs 3.33 to 6.67 ft over this record, and none of
that range is meaningful as a height above the streambed.

A constant added to stage is absorbed into the fitted zero-flow offset, and the fit
compensates by distorting the exponent, which is the parameter that controls how the
rating extrapolates to high flow. The result looks like a good fit and is wrong. The next
cell fits SBR-09 both ways.

In [ ]:
# the same station readings, interpreted two ways
raw_axis = lrc.station_sample(sbr09_station, discharge=field_discharge)   # datum ignored
wrong = lrc.fit_rating(raw_axis, method="advi")
right = lrc.fit_rating(sbr09_sample, method="advi")                       # "lowest"

print("fitted on the raw sensor axis:  R2log", round(wrong.metrics.r2_log, 5),
      " exponent b[0]", round(float(wrong.result.idata.posterior["b"].mean()), 3))
print("fitted on gage height:          R2log", round(right.metrics.r2_log, 5),
      " exponent b[0]", round(float(right.result.idata.posterior["b"].mean()), 3))
print()
print("stage axis, raw:  ", raw_axis.stage_label, raw_axis.stage_range)
print("stage axis, right:", sbr09_sample.stage_label, sbr09_sample.stage_range)
print("what was subtracted:", sbr09_sample.datum_reference_ft, "ft")

In [ ]:
# Both fits, each on the axis it was given. The scores are nearly identical and both
# curves pass through the points; the difference is in the exponent, which is what
# governs where the curve goes above the highest measurement.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, fit, title in [(axes[0], wrong, "fitted on the raw sensor axis"),
                       (axes[1], right, "fitted on gage height")]:
    fit.plot(ax=ax)
    ax.set_title(f"{title}\nR2log {fit.metrics.r2_log:.5f}")
plt.tight_layout()

Pass `stage_datum=` and the conversion happens on the way in. Each of these means
"convert my stage to gage height":

| you pass | it means |
|---|---|
| nothing | the input is already gage height |
| `612.34` | subtract this elevation (for example a USGS `alt_va`) |
| `"lowest"` | reference a point slightly below where flow would reach zero |
| a `pandas.Series` | reference elevations over time, matched to each timestamp |
| an array | one reference elevation per measurement |
| `lrc.StageDatum...` | any of the above, named and labelled |

Every conversion records what it did in `sample.datum_note` and the numeric reference in
`sample.datum_reference_ft`, so a fitted curve can be put back onto the original datum.

In [ ]:
# SBR-09 has no surveyed tie usable here, so "lowest" references the record's own low
# water. The result is a local gage height: usable for fitting and predicting at this
# sensor, not comparable across sites.
print(sbr09_sample.datum_note)
print()
print("reference subtracted:      ", round(sbr09_sample.datum_reference_ft, 3), "ft")
print("lowest measurement now at: ", round(sbr09_sample.stage_range[0], 3), "ft")
print("highest measurement now at:", round(sbr09_sample.stage_range[1], 3), "ft")

The lowest measurement does not land on 0 ft. The power-law implementation constrains
the stage of zero flow to `[0, min(stage))`, and a margin is left so that the fitted
value lands inside that interval rather than on a boundary:

- referencing at the lowest measurement leaves an empty interval, so the parameter has
  nowhere to live and the fit fails at initialization;
- referencing so that the true zero flow falls on 0, the interval's lower bound, also
  fails, because the truncated normal's unconstrained transform sends that bound to
  negative infinity and the optimizer diverges to NaN.

The second case is why the margin is measured down from the estimated stage of zero
flow rather than from the lowest measurement: any fixed offset below the lowest
measurement is exactly wrong for a dataset whose zero flow sits there.
`datum.LOWEST_MARGIN_FRACTION` documents the margin.

In [ ]:
# SBR-09's reference really does move: the sensor was lowered 1.53 ft on 2025-02-28, and
# the survey registry records it. `stepwise_reference_elevation` turns that into a
# reference elevation per reading, stepping at the timestamp of the move.
control_point = magl.control_point_elevations()["SBR-09_WL"]
moves = magl.sensor_moves()["SBR-09_WL"]
print("surveyed control point:", control_point, "ft NAVD88")
print("recorded moves:", moves)

distance_record = lrc.data.pagaia.station_series(
    sbr09_station, variable="distance", start="2024-12-01", end="2025-07-01", units="ft")
reference = lrc.data.datum.stepwise_reference_elevation(
    distance_record.index, control_point, changes=moves)

print()
print("distance down to the water:", f"{distance_record.min():.2f}",
      "to", f"{distance_record.max():.2f} ft")
print("reference elevation:      ", f"{reference.min():.2f}",
      "to", f"{reference.max():.2f} ft over", reference.nunique(), "level(s)")

In [ ]:
# Given the reference, the distance readings become water-surface elevations, and
# `station_sample` will do both steps. Note what this does NOT settle: those elevations
# land about 3 ft below the water-surface elevations the field spreadsheet records for
# the same visits - a constant offset, so the control point and the spreadsheet are not
# on the same datum. The gage-height sample is unaffected, because a constant is exactly
# what "lowest" removes, which is why this notebook fits on that.
by_distance = lrc.station_sample(sbr09_station, discharge=field_discharge,
                                 variable="distance", reference_elevation=reference,
                                 stage_datum="lowest")
print(by_distance.datum_note)
print()
print("same gage heights as the stage variable gave:",
      np.allclose(np.sort(by_distance.stage_ft), np.sort(sbr09_sample.stage_ft)))

### Distance down to the water

`distance_to_stage` performs the conversion: the water-surface elevation is the
reference point's elevation minus the distance down to the surface. Units and a moving
reference are the two things that make this error-prone in practice, and both are
arguments.

In [12]:
# a sensor whose reference point is at 615.0 ft NAVD88, reading 3.2 ft down to the water
print(float(lrc.distance_to_stage(3.2, 615.0)), "ft NAVD88")

# readings in metres, reference in feet
print(float(lrc.distance_to_stage(0.9754, 615.0, distance_units="m")), "ft NAVD88")

611.8 ft NAVD88
611.7998687664042 ft NAVD88


<a id="johnson"></a>
## 2. The stage of zero flow, and Johnson's method

A rating is a power law in effective head, not in stage:

$$Q = C\,(h - e)^{\beta}$$

`e` is the stage of zero flow: the gage height at which discharge would reach zero,
usually the streambed or the low point of the control. With the correct `e`, $\log Q$
against $\log(h - e)$ is a straight line. With the wrong `e`, the relation curves and
the fit absorbs the error into $\beta$.

Every model here estimates `e` on its own, so supplying it is optional. It is also the
parameter that a short record identifies worst, which is where a good starting point
helps most.

Johnson's three-point method solves for `e` in closed form. The selection rule is the
part most often misapplied:

1. Draw a smooth curve through the measurements.
2. Choose three discharges in geometric progression, so the middle one is the geometric
   mean of the outer two, $Q_2 = \sqrt{Q_1 Q_3}$, with $Q_1$ and $Q_3$ bracketing the
   range of interest. The spacing is in ratios of $Q$, not differences, which is why
   the middle point is not the middle measurement.
3. Read the three gage heights off the smooth curve, not off the raw points.

Because $Q = C(h-e)^\beta$, a geometric progression in $Q$ forces one in $(h - e)$:

$$(h_2 - e)^2 = (h_1 - e)(h_3 - e) \quad\Longrightarrow\quad
  e = \frac{h_1 h_3 - h_2^2}{h_1 + h_3 - 2 h_2}$$

Step 1 is what makes step 3 valid. The identity holds for points on the rating, and a
raw measurement is the rating plus error; reading $h_2$ off a noisy point sends that
error into a difference of nearly equal numbers in the denominator. On a long record
this implementation smooths with a rolling median. Below `MIN_POINTS_TO_SMOOTH`
measurements it does not, because a five-point window there would average away real
curvature rather than noise.

The two extremes are never smoothed, whatever the sample size. A median over a one-sided
window is drawn from the interior side, so it moves the highest and lowest measurements
inward on every record rather than removing noise from them. Those are $h_1$ and $h_3$,
the heights the solved offset is most sensitive to.

In [ ]:
# Johnson on SBR-09's raw sensor axis, which is where the offset is genuinely unknown.
estimate = lrc.johnson_offset(raw_axis.to_frame())
print(estimate)
print()
print("the lowest measurement sits at", round(raw_axis.stage_range[0], 4), "ft,")
print("so the solved offset is below it, as it must be.")
print()
print("the three selected points:")
print(estimate.points.to_string())

The middle point's discharge is the geometric mean of the outer two. Compare
`r2_log_raw` with `r2_log_offset`: the log-log relation goes from curved to nearly
straight once `e` is subtracted. That improvement is the diagnostic. No improvement
means either the rating was already log-linear or the offset is not identified by this
data.

In [ ]:
print("geometric mean check:",
      round(np.sqrt(estimate.points.loc["low", "discharge_cfs"]
                    * estimate.points.loc["high", "discharge_cfs"]), 4),
      "==", round(estimate.points.loc["mid", "discharge_cfs"], 4))

axes = lrc.data.zero_flow.plot_johnson(raw_axis.to_frame())
plt.tight_layout()

### Using an estimate as a prior

The power-law family places one breakpoint per segment, and the first breakpoint is the
stage of zero flow. `zero_flow=` sets what that breakpoint's prior is centred on.

In [ ]:
zero_flow_fits = {}
for setting, description in [
    (None,      "default: the power law centres its prior on Johnson's estimate"),
    ("johnson", "the same, asked for explicitly"),
    (estimate,  "the ZeroFlowEstimate object"),
    (3.0,       "a supplied number"),
    ("infer",   "no informative prior; just below the lowest measurement"),
]:
    fit = lrc.fit_rating(raw_axis, zero_flow=setting, method="advi")
    zero_flow_fits[str(setting)] = fit
    print(f"{str(setting):<12} prior centred at "
          f"{fit.result.config['zero_flow_prior_ft']:.4f} ft   # {description}")

In [ ]:
# All five on one axis. Over the measured range the prior barely matters; below the
# lowest measurement, where the offset does the work, the curves separate. On five
# points that separation is much wider than it was on the synthetic sixteen.
observed = raw_axis.to_frame()
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(observed["stage_ft"], observed["discharge_cfs"], s=45,
           facecolor="white", edgecolor="black", zorder=5, label="measurements")
for setting_name, fit in zero_flow_fits.items():
    fitted = fit.curve()
    ax.plot(fitted["stage_ft"], fitted["discharge_cfs"], lw=1.6,
            label=f"zero_flow={setting_name} "
                  f"(e={fit.result.config['zero_flow_prior_ft']:.3f})")
ax.set(yscale="log", xlabel=f"stage - {raw_axis.stage_label}",
       ylabel="discharge (cfs)", title="how the zero-flow prior moves the SBR-09 rating")
ax.grid(which="both", alpha=0.25)
ax.legend(fontsize="small")
plt.tight_layout()

bdrc treats `zero_flow` differently. For bdrc a value is not a prior mean, it fixes the
parameter. So `zero_flow=None` (the default) lets bdrc infer it under its own tuned
priors, which is normally what you want. Pass a number only when you mean to hold `c`
fixed at that value.

In [ ]:
inferred = lrc.fit_rating(sbr09_sample, model="bdrc_gplm0", method="advi")
held = lrc.fit_rating(sbr09_sample, model="bdrc_gplm0", method="advi", zero_flow=0.3)
print("bdrc, c inferred:  config zero_flow_ft =", inferred.result.config["zero_flow_ft"])
print("bdrc, c held at:   config zero_flow_ft =", held.result.config["zero_flow_ft"])

In [ ]:
# The two bdrc fits together: inferring c against holding it fixed at 0.3 ft.
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(sbr09_sample.stage_ft, sbr09_sample.discharge_cfs, s=45,
           facecolor="white", edgecolor="black", zorder=5, label="measurements")
for fit, name in [(inferred, "c inferred"), (held, "c held at 0.3 ft")]:
    fitted = fit.curve()
    ax.fill_between(fitted["stage_ft"], fitted["lower"], fitted["upper"], alpha=0.15)
    ax.plot(fitted["stage_ft"], fitted["discharge_cfs"], lw=2,
            label=f"{name}  (R2log {fit.metrics.r2_log:.4f})")
ax.set(yscale="log", xlabel=f"stage - {sbr09_sample.stage_label}",
       ylabel="discharge (cfs)",
       title="bdrc_gplm0 on SBR-09: zero_flow fixes the parameter, it is not a prior mean")
ax.grid(which="both", alpha=0.25)
ax.legend(fontsize="small")
plt.tight_layout()

<a id="diagnostics"></a>
## 3. Sampler diagnostics

Every Bayesian rating here is fitted by MCMC, and an MCMC fit can fail without raising:
the chains return numbers either way. Check convergence before reading anything else,
because a fit whose sampler did not converge has neither trustworthy parameters nor a
trustworthy ELPD. These say nothing about whether the *model* is right — only whether
the sampler drew from the posterior it was given.

**R-hat**, the Gelman-Rubin statistic. Four chains start from different points. If they
found the same distribution, the variance between chains matches the variance within
each one and their ratio is 1. Above about 1.01 the chains disagree.

**Effective sample size.** MCMC draws are correlated, so 4000 draws are worth fewer than
4000 independent ones; ESS is how many they are worth, per parameter, estimated from how
fast the autocorrelation decays.

- `ess_bulk` — the centre of the distribution: the mean and sd.
- `ess_tail` — the 5th and 95th percentiles, and therefore the credible interval.

So good bulk and poor tail ESS means a trustworthy curve and an untrustworthy band. The
threshold `settings.ESS_GOOD` = 400 is 100 per chain across the default four: a floor
below which the ESS estimate is itself too noisy, not a target.

In [ ]:
# The full ArviZ summary for the best-scoring model, which is where every diagnostic in
# this section comes from. `diagnostics()` is this table, cut down and sorted.
best = comparison.best
summary = lrc.evaluate.diagnostics.posterior_summary(best)
print(best.name, "-", len(summary), "rows")
summary.head(10)

### Reading the summary table

`posterior_summary` is `arviz.summary`: one row per scalar quantity in the posterior.

**The left-hand index column** is the parameter name. Vectors and matrices are unpacked
one row per element, position in brackets, zero-based: `b[0]` is the first entry of `b`,
`X[1, 7]` is row 1 column 7 of `X`. That is why the table is longer than the model has
parameters, and why its length depends on the number of measurements.

| name | family | what it is |
|---|---|---|
| `a` | ratingcurve | intercept, on standardized log discharge |
| `b[i]` | ratingcurve | exponent of segment `i`. `b[0]` is the exponent; `b[i>0]` is the *change* in slope at breakpoint `i` |
| `hs_[i, 0]` | ratingcurve | sampled breakpoints, before sorting |
| `hs[i, 0]` | ratingcurve | *deterministic*: the same, sorted. `hs[0, 0]` is the stage of zero flow |
| `sigma` | ratingcurve | residual scatter, on standardized log discharge |
| `X[i, j]` | ratingcurve | *deterministic*: design matrix, `log(max(h - hs[i], 0) + ho[i])` at measurement `j` |
| `w[k]` | spline | weight on spline basis function `k` |
| `theta[i]` | bdrc | the sampled hyperparameter vector; the four rows below are its entries under readable names, so every bdrc parameter appears twice |
| `zeta` | bdrc | `log(h_min - c)`, so zero flow is `c = h_min - exp(zeta)` |
| `log_sigma_eps2` | bdrc | log error variance |
| `log_sigma_beta` | bdrc | log of how far the exponent may wander from constant |
| `log_phi_beta` | bdrc | log of the stage range over which it wanders |

Deterministic rows get R-hat and ESS like the rest, but there is nothing to tune in them:
fix the parameter they are computed from. bdrc has no `a` or `b` row because it draws
those in closed form rather than by MCMC.

In [ ]:
# Every name in the index, grouped: how many rows each parameter contributes, and the
# shape it has in the posterior. This is the map of the left-hand column.
posterior = best.result.idata.posterior
base_name = summary.index.str.replace(r"\[.*\]", "", regex=True)

pd.DataFrame({
    "rows in the summary": base_name.value_counts(),
    "shape per draw": pd.Series({name: str(tuple(posterior[name].shape[2:]) or "scalar")
                                 for name in posterior.data_vars}),
}).sort_values("rows in the summary", ascending=False)

**The columns**

| column | what it is |
|---|---|
| `mean`, `sd` | posterior mean of the parameter, and its posterior standard deviation |
| `hdi_3%`, `hdi_97%` | 94% highest-density interval: the narrowest interval holding 94% of the posterior. On a *parameter* — `rating.interval()` is the interval on *discharge* |
| `mcse_mean`, `mcse_sd` | Monte Carlo standard error: how much the reported mean and sd would move on a rerun with another seed |
| `ess_bulk`, `ess_tail` | effective draws for the centre, and for the tails |
| `r_hat` | between-chain agreement |

`sd` and `mcse_mean` are different uncertainties: `sd` is what the data leave
undetermined and does not shrink with more draws; `mcse_mean` is sampling noise and
shrinks as `sd / sqrt(ess)`. When it is a small fraction of `sd`, more draws buy nothing.

Read `r_hat`, then `ess_bulk`, then `mean`. `diagnostics()` returns exactly those
columns, worst-first.

In [ ]:
print(lrc.evaluate.diagnostics.convergence_report(best))
best.diagnostics().head(8)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
lrc.evaluate.diagnostics.plot_convergence(best, ax=axes[0])
lrc.evaluate.diagnostics.plot_effective_sample_size(best, ax=axes[1])
plt.tight_layout()

Across every model at once. A model that failed to converge should not be ranked against
the others, whatever its scores look like.

In [ ]:
lrc.evaluate.diagnostics.convergence_table(comparison)

### The trace plot

`plot_trace` is `arviz.plot_trace`. **Two panels per parameter, one colour per chain:**

- **left, the marginal density** — a KDE of each chain's draws. Four curves on top of
  each other means the chains agree. Four curves in different places is what R-hat > 1.01
  looks like.
- **right, the trace** — draw number against parameter value: the sampler's actual path.
  Healthy is a flat band of noise, dense and even, chains indistinguishable. The three
  failures it shows are a **trend** (tuning too short), a **flat step** where the chain
  stopped moving, and one chain **separated** in a band the others never visit.

Pass `var_names` unless the model is small: `X` has one entry per measurement, so an
unfiltered call on a gage buries the parameters you wanted under a hundred panels of a
deterministic quantity.

In [ ]:
# The model's own parameters: everything except the design matrix, and, for bdrc, the
# `theta` vector whose entries are already listed under their own names.
shown = [name for name in best.result.idata.posterior.data_vars
         if name not in {"X", "theta", "hs_"}]
print("plotting:", shown)

axes = lrc.evaluate.diagnostics.plot_trace(best, var_names=shown,
                                           figsize=(11, 2.1 * len(shown)))
axes[0, 0].figure.suptitle(f"{best.name}: density (left) and trace (right), one colour "
                           f"per chain", y=1.0)
plt.tight_layout()

A converged trace is meant to look like nothing, so here are both shapes side by side
with their numbers attached. The unconverged parameter still reports a perfectly
reasonable mean and sd; only `r_hat` and the ESS show it is an average over chains that
never agreed.

In [ ]:
import arviz as az

# Two fabricated parameters, four chains of 1000 draws each. Nothing is being fitted
# here; the point is to see the two shapes side by side with their numbers attached.
demo_rng = np.random.default_rng(7)

# well mixed: independent draws from the same distribution in every chain
well_mixed = demo_rng.normal(0.0, 1.0, (4, 1000))

# not mixed: a slow random walk, each chain starting somewhere different and never
# getting far enough to meet the others
walk = np.cumsum(demo_rng.normal(0.0, 0.05, (4, 1000)), axis=1)
separated = walk + np.array([[-3.0], [-1.0], [1.0], [3.0]])

demo = az.from_dict({"well_mixed": well_mixed, "separated": separated})
print(az.summary(demo, round_to=3)[["mean", "sd", "ess_bulk", "ess_tail", "r_hat"]])

az.plot_trace(demo, figsize=(11, 4.5))
plt.suptitle("what converged (top) and not converged (bottom) look like", y=1.02)
plt.tight_layout()

### Two plots that sharpen the trace

Overlapping four lines hides things. These separate what the trace mixes together: the
rank plot is R-hat drawn out, the autocorrelation plot is ESS drawn out.

- **Rank plot** — pool all draws, rank them, histogram each chain's ranks. Chains
  sampling the same distribution each hold an equal share of every rank, so every
  histogram is flat. A chain sitting high takes the top ranks and tilts.
- **Autocorrelation** — how much a draw resembles the one `k` steps back. Dropping to
  zero within a few lags means nearly independent draws and a high ESS; still high at
  lag 20 means each draw adds little.

In [ ]:
import arviz as az

# Rank plot: pool every draw from every chain, rank them, and histogram each chain's
# ranks. Flat and equal across chains is what convergence looks like here.
az.plot_rank(best.result.idata, var_names=shown, kind="bars", figsize=(11, 3.2))
plt.suptitle(f"{best.name}: rank plot - each chain should own its share of every rank",
             y=1.02)
plt.tight_layout()

# Autocorrelation: how far apart two draws must be before they stop resembling each
# other. The faster this falls to zero, the higher the effective sample size.
az.plot_autocorr(best.result.idata, var_names=shown, combined=True, figsize=(11, 3.2))
plt.suptitle(f"{best.name}: autocorrelation by lag - this is what ESS summarizes", y=1.02)
plt.tight_layout()

### Divergences, and where a low ESS comes from

A **divergent transition** is a step where NUTS's trajectory blew up, in a region of
sharp curvature the step size could not follow. It marks parameter space the sampler
could not enter, so the draws are biased away from it — and that can happen at a
perfectly acceptable R-hat.

The pair plot is where they become readable: two parameters per panel, divergences
marked. It also explains a low ESS. A long diagonal cloud means the data constrain the
*combination* of two parameters but not each one, and a sampler crossing a narrow
diagonal ridge takes many small steps. `a` and `b[0]` do this whenever the stage of zero
flow is poorly identified: raising the exponent and lowering the intercept gives back
nearly the same curve.

In [ ]:
import arviz as az

sample_stats = getattr(best.result.idata, "sample_stats", None)
if sample_stats is not None and "diverging" in sample_stats:
    print("divergent transitions:", int(sample_stats["diverging"].sum()),
          "of", int(sample_stats["diverging"].size), "draws")
    az.plot_pair(best.result.idata, var_names=shown, divergences=True,
                 figsize=(9, 9), scatter_kwargs={"alpha": 0.15, "s": 8})
    plt.suptitle(f"{best.name}: how the parameters trade off against each other", y=1.0)
    plt.tight_layout()
else:
    print("no sample_stats on this fit, so there are no divergences to count.\n"
          "That means it was fitted with ADVI, not NUTS.")

`method="advi"` replaces sampling with a variational approximation. It is much faster and
adequate for a first look, but it produces one chain, so there is no between-chain
variance to compute and no ELPD or Pareto-k. An empty predictive-score column in a
comparison table usually means the fits were ADVI.

In [ ]:
quick = lrc.fit_rating(sbr09_sample, method="advi")
print("ADVI elpd_loo:", quick.metrics.elpd_loo)
print("ADVI pareto_k:", quick.pareto_k()["pareto_k"].to_numpy()[:3])
print()
print("NaN here means the score was not computed, not that the fit is unreliable.")

In [ ]:
# ADVI against the full MCMC fit of the same model on SBR-09. The curve is close; the
# band is what the approximation gets wrong, and it is usually too narrow.
fig, ax = plt.subplots(figsize=(9, 5))
for fit, name, color in [(sbr09, "NUTS", "#4c72b0"), (quick, "ADVI", "#dd8452")]:
    fitted = fit.curve()
    ax.fill_between(fitted["stage_ft"], fitted["lower"], fitted["upper"],
                    color=color, alpha=0.18)
    ax.plot(fitted["stage_ft"], fitted["discharge_cfs"], color=color, lw=2, label=name)
ax.scatter(sbr09_sample.stage_ft, sbr09_sample.discharge_cfs, s=45,
           facecolor="white", edgecolor="black", zorder=5, label="measurements")
ax.set(yscale="log", xlabel=f"stage - {sbr09_sample.stage_label}",
       ylabel="discharge (cfs)", title="SBR-09: same model, two fitting methods")
ax.grid(which="both", alpha=0.25)
ax.legend(fontsize="small")
plt.tight_layout()

<a id="cv-depth"></a>
## 4. Cross-validation, in depth

`elpd_loo` approximates leave-one-out from one fit (`psis.ipynb`). Cross-validation does
it literally: refit without some measurements, then predict them. It is slower, and it
is the one to believe when the two disagree. The setup cell has already run it on
SBR-09; this is what its numbers mean, and how to score the part of the curve that
matters most.

Two details the implementation handles:

- Under leave-one-out, read the pooled scores. NSE divides by the variance of the
  held-out observations, and a single point has none, so a per-fold NSE is NaN by
  construction. Pooling every fold's single prediction into one vector of length `n` and
  scoring that once is what "leave-one-out cross-validated NSE" normally means.
- Do not stratify a leave-one-out split. Stratification exists to stop a small random
  training draw from missing part of the stage range. Applied to `n-1` training points it
  does the opposite: most bins hold one point, which is therefore always selected for
  training, so the high-flow measurements are never tested.

Folds are fitted with ADVI by default, since a sweep is many refits and its purpose is to
show how the curve moves rather than to produce an ELPD. A table placing `test_nse` next
to `elpd_loo` is therefore mixing two fitting regimes: ADVI for the folds, NUTS for the
in-sample fit, and section 3 above is where that difference is drawn.

### The holdout scheme, on a USGS gage

The run above left one measurement out at a time, because SBR-09 has a handful of them.
There are two schemes, because sample sizes differ by an order of magnitude:

- Tens of measurements, as at a USGS gage, use `"holdout"`: train on a fraction of the
  sample and predict the rest, over several stage-stratified draws. `holdout=0.4` below
  trains on twelve of the twenty measurements and predicts the other eight.
- A handful, as at a MAGL sensor, use `"loo"`: leave one out at a time, `n` folds,
  training on `n-1`.

`scheme="auto"` picks by sample size, and its default holdout is a deliberately demanding
90% - which on twenty measurements would train on two, fewer than a power law needs. That
is the reason to set `holdout` explicitly on a sample this size.

Every fold is a refit, so this is the slowest cell in these notebooks: two models over
four splits is eight fits. Adding `bdrc_gplm0` to the list roughly triples that.

In [ ]:
usgs_sample = lrc.gage_sample("06893578", start="2024-01-01")
usgs_cv = lrc.cross_validate(usgs_sample, models=["power_law", "spline"],
                             scheme="holdout", holdout=0.4, n_splits=4)
print(usgs_cv)
print(usgs_cv.scheme["description"])
usgs_cv.headline

In [ ]:
# Every fold's curve on one axis, per model. The spread between folds answers "how much
# does this rating depend on which measurements I happened to have?" - a tight bundle is
# a model that recovers the same rating from any subset.
for model_name in usgs_cv.curves_by_model():
    ax = usgs_cv.plot(model=model_name)
    ax.set_title(f"USGS 06893578, {model_name}: one curve per holdout draw")
plt.tight_layout()

In [ ]:
usgs_cv.per_fold

### Extrapolating to high flow

The high end is where measurements are sparsest, where the consequences of error are
largest, and where a flexible model that fits well in the middle can be badly wrong. An
overall score hides this, because low-flow measurements outnumber high-flow ones.

`high_flow_skill` scores only the held-out predictions above a stage quantile and reports
predicted over observed, which is interpretable in a way that RMSE in cfs is not. A ratio
of 1.0 is unbiased; 0.7 means under-predicting a flood by 30%.

In [ ]:
high_flow = lrc.evaluate.crossval.high_flow_skill(cv, quantile=0.6)
print(f"held-out measurements at stage >= {high_flow.attrs['stage_threshold_ft']:.2f} ft")
high_flow

In [ ]:
ax = lrc.view.plots.plot_high_flow_error(cv)

<a id="magl"></a>
## 5. The same sensor, from the MAGL files

Everything specific to the LimnoTech MAGL network lives in one module; nothing else in
the package knows about it. The setup cell read SBR-09 live from pagaia. The same sensor can
be read from the MAGL files instead, which is what to do when the VPN is unavailable and
what the batch runs use.

The two routes do not give the same sample, and the difference is worth knowing before
trusting either:

| | pagaia (section 2) | the files (here) |
|---|---|---|
| stage | the station's own reading, matched to each visit within 3h | the water-surface elevation the field crew recorded at the visit |
| n | 5 - two visits have no station reading in range | 7 - every visit has a field stage |
| datum | depth above the sensor's zero, then `"lowest"` | NAVD88 ft, then `"lowest"` |

They agree on the span of the record, 3.34 ft against 3.29 ft, which is the check that
says the two are measuring the same thing.

MAGL is the hard case either way, for two reasons:

- The sensors do not measure stage. They report distance down to the water, so stage is
  the reference elevation minus that distance. The reference also moves: SBR-09 was
  lowered 1.53 ft on 2025-02-28, and the reference has to step with it at that timestamp
  or the stage record is discontinuous.
- There are three to seven discharge measurements per sensor, which is why
  `magl.cross_validate` defaults to leave-one-out.

In [ ]:
magl.list_sensors()          # which sensors have enough measurements to fit

In [ ]:
magl.discharge_report("SBR-09")       # the curated manual-discharge spreadsheet

In [ ]:
magl_sample = magl.sensor_sample("SBR-09")     # stage_datum="lowest" by default
print(magl_sample)
print(magl_sample.datum_note)
magl_sample.support()

In [ ]:
magl_comparison = magl_sample.compare(models=["power_law", "bdrc_gplm0"])
magl_comparison.metrics

In [ ]:
# Three to seven measurements is what a MAGL rating is fitted on, and the plot is the
# honest picture of that: the two families agree where the points are and diverge
# immediately outside them.
ax = magl_comparison.plot()
ax.set_title("SBR-09: two families on a handful of measurements")
plt.tight_layout()

In [ ]:
magl_cv = magl.cross_validate(magl_sample, models=["power_law", "bdrc_gplm0"])
print(magl_cv.scheme["description"])
magl_cv.pooled                    # pooled, because per-fold NSE is undefined here

In [ ]:
# Leave-one-out on this many points is a severe test, and the fold curves show why: each
# refit drops one measurement out of a handful, and the rating moves visibly.
ax = magl_cv.plot()
ax.set_title("SBR-09: leave-one-out folds")
plt.tight_layout()

### The co-located check

The sharpest available test of the MAGL stage derivation is to pair USGS-measured
discharge with the co-located MAGL sensor's stage, tied to the gage's datum. The gage's
rating is already known, so any disagreement is attributable to the stage rather than to
the discharge.

The tie is `gage height = NAVD88 elevation - alt_va`, which is valid only if the gage's
datum is also NAVD88. When it is not, the result is an empty sample with the reason
attached rather than a plausible-looking wrong answer.

In [ ]:
colocated = magl.colocated_sample("SR-08")
print(colocated)
print(colocated.skipped or colocated.datum_note)

### The two routes, side by side

The file route and the API route are two measurements of the same rating, so putting
them on one axis is the available check on both. What should agree is the shape; what
will not is the absolute stage, because each route references a different zero and
`"lowest"` then removes a different constant.

In [ ]:
# the same sensor, both ways
file_rating = lrc.fit_rating(magl_sample, method="advi")
api_rating = lrc.fit_rating(sbr09_sample, method="advi")

fig, ax = plt.subplots(figsize=(9.5, 5.5))
for fit, sample, name, color in [
        (file_rating, magl_sample, f"MAGL files (n={len(magl_sample)})", "#4c72b0"),
        (api_rating, sbr09_sample, f"pagaia (n={len(sbr09_sample)})", "#dd8452")]:
    curve = fit.curve()
    ax.plot(curve["stage_ft"], curve["discharge_cfs"], color=color, lw=2,
            label=f"{name}  R2log {fit.metrics.r2_log:.3f}")
    ax.scatter(sample.stage_ft, sample.discharge_cfs, s=45, facecolor="white",
               edgecolor=color, zorder=5)
ax.set(yscale="log", xlabel="local gage height (ft)", ylabel="discharge (cfs)",
       title="SBR-09: the same rating from the files and from the API")
ax.grid(which="both", alpha=0.25)
ax.legend(fontsize="small")
plt.tight_layout()

In [ ]:
# The two routes put the same discharge at different gage heights, because the datums
# differ. Compare them where it is meaningful: at the same discharge, how far apart are
# the stages the two ratings imply?
for flow in [1.0, 5.0, 20.0, 60.0]:
    stages = {}
    for name, fit in [("files", file_rating), ("pagaia", api_rating)]:
        curve = fit.curve()
        stages[name] = float(np.interp(flow, curve["discharge_cfs"],
                                       curve["stage_ft"], left=np.nan, right=np.nan))
    print(f"Q = {flow:>5.1f} cfs   files {stages['files']:.3f} ft   "
          f"pagaia {stages['pagaia']:.3f} ft   difference "
          f"{stages['files'] - stages['pagaia']:+.3f} ft")

<a id="map"></a>
## 6. The map

The interactive map is the main output of a batch run, because the useful question is
usually not what one curve looks like but which model wins where, and whether the answer
changes between sites.

Hovering a site shows every fitted curve with its score table, and, when
cross-validation was run, a model selector and fold arrows for stepping through each
refit. Axes are frozen per site, so a curve that moved is visible as having moved instead
of being rescaled out of view. Hovering an individual measurement shows its Pareto-k
under each model.

In [ ]:
report = lrc.report(models=["power_law", "bdrc_gplm0"], sites=["SBR"],
                    cross_validate=True)
print(report)
report.results.head(10)

In [ ]:
report.cv_table()

In [ ]:
# The map is a standalone HTML file. Open it in a browser; it does not render usefully
# inside a notebook cell.
print(report.map_path)

The same run from the command line is in section 8.


<a id="many"></a>
## 7. Fitting many sites in parallel

`lrc.report(...)` above is the MAGL pipeline. `lrc.fit_many` is the general one: give it
samples and model keys and it fans out across processes, one task per (site, model),
longest job first.

The return value is built from files rather than from live objects, because a fitted
model cannot cross a process boundary: the spline carries a patsy `DesignInfo` that will
not pickle, and neither a PyMC model nor an `InferenceData` survives being sent between
processes. Each worker writes its own manifest and posterior, and the collection is
assembled from those files. Reloading a sweep from disk therefore gives an object of the
same shape.

In [ ]:
# The four MAGL sensors with the most measurements, read from the files so this runs
# without the VPN. Real sites, real sample sizes: 5 to 7 measurements each.
sites = {sensor: magl.sensor_sample(sensor)
         for sensor in ["SBR-09", "SBR-10", "SBR-14", "SBR-13"]}
for name, sample in sites.items():
    print(f"{name}  n={len(sample):<3} discharge {sample.discharge_range[0]:.2f}"
          f" to {sample.discharge_range[1]:.1f} cfs")

# workers=1 because on Windows a process pool requires the call to be inside
# `if __name__ == "__main__":`, which a notebook cell is not. In a script, omit `workers`
# and it uses one per core.
results = lrc.fit_many(sites, models=["power_law", "bdrc_gplm0"],
                       directory="fits", workers=1, method="advi")
results

In [ ]:
# one row per (site, model), best predictive score first
results.metrics[["site", "model", "n", "nse", "r2_log", "elpd_loo", "seconds"]]

In [ ]:
# One panel per sensor, both models on each. Axes are per-site, because the point of a
# sweep is to see which family wins where, not to compare discharge between streams.
# `results[...]` hands back a SavedRating, so `curve` is a stored table, not a method.
fig, axes = plt.subplots(1, len(sites), figsize=(4.6 * len(sites), 4.2))
for ax, site_name in zip(np.atleast_1d(axes), sites):
    for model_name in sorted(results[site_name]):
        fitted = results[site_name, model_name].curve
        ax.plot(fitted["stage_ft"], fitted["discharge_cfs"], lw=1.6, label=model_name)
    measured = sites[site_name].to_frame()
    ax.plot(measured["stage_ft"], measured["discharge_cfs"], "o",
            color="black", ms=4, label="measurements")
    ax.set(yscale="log", xlabel="gage height (ft)", ylabel="discharge (cfs)",
           title=f"{site_name} (n={len(measured)})")
    ax.grid(which="both", alpha=0.25)
    ax.legend(fontsize="small")
plt.tight_layout()

In [ ]:
# the collection is indexed by site and by (site, model)
print("models fitted for SBR-09:", sorted(results["SBR-09"]))

one = results["SBR-09", "power_law"]
print(one)
print("  Q(2 ft) =", round(one.predict(2.0), 2), "cfs")
print("  95% interval:", tuple(round(v, 2) for v in one.interval(2.0)))
print("  fitted with:", one.sampler.get("method"), "/", one.sampler.get("nuts_sampler"))
print("  best for SBR-09:", results.best("SBR-09"))

In [ ]:
# anything that did not fit reports why, rather than being dropped
results.failures

<a id="cli"></a>
## 8. The command line

Every fit above has a command-line equivalent, and the reading commands go further than
that: `list` and `inspect` read manifests only, so they run where nothing can be fitted -
no PyMC, no ArviZ, no compiler. Fit on the machine with the toolchain, read anywhere.

```
rating-curves --site SBR --models power_law bdrc_gplm0 --cv -v    fit a MAGL site
rating-curves --data my_measurements.csv --stage-datum lowest     fit a CSV
rating-curves --list-sites
rating-curves --list-models
rating-curves export output/fitted_curves --to ship/     a portable copy plus the CSVs
rating-curves list ship/                                 one row per rating
rating-curves inspect ship/SBR-09__power_law.rating.json --curve
```


In [ ]:
!rating-curves list fits
